In [6]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding


In [ ]:
LLM_MODEL = "microsoft/deberta-v3-small"

In [7]:

# Data loading + preprocessing
csv_path = "data/train.csv"
df = pd.read_csv(csv_path)

# 1) Label encoding
if set(["winner_model_a", "winner_model_b", "winner_tie"]).issubset(df.columns):
    df["label"] = (
        df["winner_model_a"] * 0 +
        df["winner_model_b"] * 1 +
        df["winner_tie"] * 2
    )
else:
    raise ValueError("Missing one of winner_model_a/winner_model_b/winner_tie")

# 2) Build text field
if set(["prompt", "response_a", "response_b"]).issubset(df.columns):
    df["text"] = (
        "Prompt:\n" + df["prompt"].astype(str) +
        "\n\nResponse A:\n" + df["response_a"].astype(str) +
        "\n\nResponse B:\n" + df["response_b"].astype(str)
    )
else:
    raise ValueError("Missing one of prompt/response_a/response_b")

print("Loaded", len(df), "rows")
print(df["label"].value_counts())

Loaded 57477 rows
label
0    20064
1    19652
2    17761
Name: count, dtype: int64


In [ ]:


# 3) HF dataset
dataset = Dataset.from_pandas(df[["text", "label"]])

# 4) split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]


def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )


tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


Map: 100%|██████████| 5748/5748 [00:00<00:00, 5949.11 examples/s]


In [ ]:


model = AutoModelForSequenceClassification.from_pretrained(
    LLM_MODEL,
    num_labels=3
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 72770.71it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier

In [11]:


training_args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    num_train_epochs=1,
    fp16=False,
    push_to_hub=False,
    report_to="none",
    # log every 1000 steps:
    do_eval=False,
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=1000,  
)


In [12]:


data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1000,0.537741
2000,0.000000
3000,0.000000
4000,0.000000
5000,0.000000
6000,0.000000
7000,0.000000
8000,0.000000
9000,0.000000
10000,0.000000


In [ ]:
# Save the trained model and tokenizer
trainer.save_model("./model")
tokenizer.save_pretrained("./model")

# Evaluate and show metrics
metrics = trainer.evaluate(eval_dataset=val_dataset)
print("Validation metrics:", metrics)

# Predict to verify output shape
predictions = trainer.predict(val_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)
print("Sample predictions:", pred_labels[:10])